## TC 5033
## Deep Learning
## Transformers

#### Activity 4: Implementing a Translator

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



#### Script to convert csv to text file 

In [1]:
#This script requires to convert the TSV file to CSV
# easiest way is to open it in Calc or excel and save as csv
import os
import pandas as pd
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "2.0"

current_dir = os.getcwd()
PATH = os.path.join(current_dir,"data/eng-spa2026.csv") #   current_dir+'/data/eng-spa2026.csv'
print(PATH)
df = pd.read_csv(PATH, sep=',')

/Users/ricardovargas/Documents/repositorios/python/DL_fundamentals/data/eng-spa2026.csv


In [2]:
df.shape

(280038, 4)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 280038 entries, 0 to 280037
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   1276                  280038 non-null  int64
 1   Let's try something.  280038 non-null  str  
 2   2481                  280038 non-null  int64
 3   ¡Intentemos algo!     280038 non-null  str  
dtypes: int64(2), str(2)
memory usage: 8.5 MB


In [4]:
eng_spa_cols = df.iloc[:, [1, 3]]
eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()  
eng_spa_cols = eng_spa_cols.sort_values(by='length')  
eng_spa_cols = eng_spa_cols.drop(columns=['length'])  

output_file_path = os.path.join(current_dir,'data/eng-spa4.txt')
eng_spa_cols.to_csv(output_file_path, sep='\t', index=False, header=False)

## Transformer - Attention is all you need

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu" )
print(device)

mps


In [7]:
MAX_SEQ_LEN = 128

In [8]:
class PositionalEmbedding(nn.Module):
    """
        This class acts like a layer that will calculates and assigns the corresponding sigmoid signal
        to the embeddings, so the transformer can determine what's the position of the embedding within the 
        sentences.
    """
    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        """
            This is the constructor and below the attributes.

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                max_seq_len (int): Max number of tokens that a sentence will have, it defaults to the constant MAX_SEQ_LEN.
        """
        super().__init__()
        # It creates a torch with zeros of dimensions [max_seq_len, d_model]
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        # It create a temp array with the length of the max sequence and then add a new dimension 
        # making any element of the initial array become a row 
        # torch.arange(0, max_seq_len, dtype = torch.float) ==> [ 1, 2, 3, 4, 5]
        # .unsqueeze(1) ==> [ [1], [2], [3], [4]...]
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        # 
        div_term = torch.exp(torch.arange(0, d_model, 2).float() 
                             * (-math.log(10000.0)/d_model))
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term) # Sine of the token position by the div_term for event positions
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term) # Cosine of the token position by the div_term for odd
        # pos_embed_matrix before unsqueezing it has these dimensions [max_seq_len, d_model]
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)
        # pos_embed_matrix after unsqueezing it has these dimensions [max_seq_len, new_dim, d_model]
        # This last step should be similar to do this self.pos_embed_matrix.unsqueeze(1) as we add the additional dimension in the second index.
        
    def forward(self, x):
#         print(self.pos_embed_matrix.shape)
#         print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    """
        Class that implements the multihead attention.
    """
    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'
        
        # d_model = 512, num_heads = 8
        self.d_v = d_model // num_heads # d_v = 64

        self.d_k = self.d_v
        self.num_heads = num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        '''
        Q, K, V -> [batch_size, seq_len, num_heads*d_k]
        after transpose Q, K, V -> [batch_size, num_heads, seq_len, d_k]
        '''
        # With the view instruction we reshape the information in a way that the size
        # of the second dimension/index is inferred based on the other three dimensions.
        # Given that the input Q has these dimensions [batch_size, seq_len, d_model] = 64, 128, 512 
        # After the view it should have something like [64, 128 , 8 , 64]
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        # After the transpose it will have this [64, 8, 128, 64], matching with the comments the professor left.
        
        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)
        
        return weighted_values, attention
        
        
    def scale_dot_product(self, Q, K, V, mask = None):
        """
            This functions basically, calculates the score attention. If follows the google doc 

            * it does a multiplication of Q by K (Transpose) and applies the division by the square root of d_k.
            * Transform the score into probabilities using softmax, but first it converts to tiny values (close to zero), using the defined mask. So softmax 
              assigns a small probability.
            * A new multiplication of the calculated probabilities by the V values that were obtained for V (layer).

            Attributes:

                Q (torch.Dataset): It's the batch data that should be processed and processed by the Query layer.
                K (torch.Dataset): It's the batch data that should be processed and processed by the Key layer.
                V (torch.Dataset): It's the batch data that should be processed and processed by the Value layer.
                mask (torch.Dataset): It's the set of Trues and Falses that are used as a mask. The purpose is to control 
                the Transformer doesn't see future or unseen words. In other words, the softmax function calculates a probabilistic 
                distribution given the whole set of words, with this we are assigning tiny values to the initial "tokens" and the unseen 
                words for the english words.

            Returns: 
               weighted_values and attention scores
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9) # I'm not sure but I think the mask does nothing for the english input, but for spanish it assigns a low score.
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)
        
        return weighted_values, attention
        

class PositionFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))
    
class EncoderSubLayer(nn.Module):
    """
        This class is used to create the logic that one encoder layer have.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        """
            This is the constructor where the next level of layers are defined

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                num_heads (int): Number of heads that are running in parallel 
                d_ff (int): number of feed forward layers
                dropout (float)=0.1: This parameter controls the probability of assign a zero to a position to the attention_score. This parameters is used by
                the Dropout layer. The purpose of the dropout is to regularize the model helping to reduce the overfitting. 
        """
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask = None):
        """
            This method calls the self_attention method that calculates the attention score. 
            And then execute the forward for the other layers.
        """
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    """
        This class is used to create the encoder layer.
        It is made of a set of Encoder Sub Layers and a normalization layer.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        """
            This is the constructor and below the attributes.

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                num_heads (int): Number of heads that are running in parallel (TODO: we will understand this in detail later)
                d_ff (int): number of feed forward layers
                num_layers (int): Number of layers (encoders and decoders to be created)
                dropout (float)=0.1 TODO: To be defined.
        """
        super().__init__()
        # Define an a list of layers 
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # Define a normalization layer
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        """
            This method execute the forward step for the list of layers and applies the normalization layer as a last step.
            
            x (torch.Tensor): it's one of the batches that will be processed.
            mask (torch.Tensor): It's the set of Trues and Falses that are used as a mask.

        """
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):
    """
        This class is used to create the logic that one decoder layer have.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
            This is the constructor where the next level of layers are defined

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                num_heads (int): Number of heads that are running in parallel 
                d_ff (int): number of feed forward layers
                dropout (float)=0.1: This parameter controls the probability of assign a zero to a position to the attention_score. This parameters is used by
                the Dropout layer. The purpose of the dropout is to regularize the model helping to reduce the overfitting. 
        """
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)
        
        # Here is the most important part of the magic.
        # We are passing as (Query) the attention score for spanish encoding (X) after droppingout and normalizing 
        # We are passing as (Key and Value) the encoder_output (weights from the encoder for english words)
        # Just the last parameter the mask
        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)
        
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)
        
class Decoder(nn.Module):
    """
        This class is used to create the decoder layer.
        It is made of a set of Decoder Sub Layers and a normalization layer.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        """
            This is the constructor and below the attributes.

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                num_heads (int): Number of heads that are running in parallel (TODO: we will understand this in detail later)
                d_ff (int): number of feed forward layers
                num_layers (int): Number of layers (encoders and decoders to be created)
                dropout (float)=0.1 TODO: To be defined.
        """
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, encoder_output, target_mask, encoder_mask):
        """
            This method is responsible for implementing the forward logic, what it's important here is that this function 
            is the point of connection btw the encoder and decoder. As it's evident in the parameters this functions receives
            the encode output.

            Attributes:

                x (torch.Tensor): English data
                encoder_ouput (torch.Tensor): Weights that were determined by the encoder.
                target_mask (torch.Tensor): Masks that is used ignore future words in spanish.
                encoder_mask (torch.Tensor): Masks that was used to ignore padding words in english words.
        """
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

In [9]:
class Transformer(nn.Module):
    """
        The class transformer works as an entry point and main layer for the model that encapsulates
        all the logic to replicate the transformer architecture of the transformer that was published in 2017.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size, 
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        """
            This is the constructor and below the attributes.

            Attributes:

                d_model (int): It's the size or number of features of the embeddings.
                num_heads (int): Number of heads that are running in parallel (TODO: we will understand this in detail later)
                d_ff (int): number of feed forward layers
                num_layers (int): Number of layers (encoders and decoders to be created)
                input_vocab_size (int): Size of dictionary of source/input language
                target_vocab_size (int): Size of dictionary of target/output language
                max_len (int): Max number of tokens that a sentence will have, it defaults to the constant MAX_SEQ_LEN.
                dropout (float)=0.1 TODO: To be defined.
        """
        super().__init__()
        # It creates the embeddings layers for english and spanish words/tokens.
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        # It creates a layer that will calculates the positional information for the embeddings.
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)
        
    def forward(self, source, target):
        """
            The forward method that execute what the steps are thus the transformer can calculate the parameters.

            Attributes: 
                source (Dataset): The dataset with the input information in this case it will be the sentences in english.
                target (Dataset): The dataset with the expected output that it will be used as the expected translated language, 
                                  in this case it will be the sentences in spanish
        """
        # Encoder mask
        source_mask, target_mask = self.mask(source, target)
        # Embedding and positional Encoding
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)
        
        # Decoder embedding and postional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        # Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)
        
        return self.output_layer(output)
        
        
    def mask(self, source, target):
        """
            This function creates a mask that helps to ignore future and padding tokens. 
            The index cero will have the padding token and this token will be ignore for both inputs source and target.
            However, for the target dataset the future tokens are ignored, to make that possible
            an array is created with (true and false values) with an structure similar to the one below.
            [
               [True, False, False, False],
               [True, True,  False, False],
               [True, True,  True,  False],
               [True, True,  True,  True]
            ]
            In this way we can control the model doesn't pay attention to the unseen words.

            Attributes: 
                source (Dataset): The dataset with the input information in this case it will be the sentences in spanish.
                target (Dataset): The dataset with the expected output that it will be used as the expected translated language, 
                                  in this case it will be the sentences in english
        """
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2) # Adds one dimension on index 1, and then another dimension on index 2.
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask
        return source_mask, target_mask
        

#### Simple test

In [10]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [11]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size, 
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [12]:
output = model(source, target)

In [13]:
# # Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


### Translator Eng-Spa

In [14]:
FORMATTED_DATA_PATH = os.path.join(current_dir, 'data/eng-spa4.txt')
FORMATTED_DATA_PATH

'/Users/ricardovargas/Documents/repositorios/python/DL_fundamentals/data/eng-spa4.txt'

In [15]:
with open(FORMATTED_DATA_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

In [16]:
eng_spa_pairs[:10]

[['Go!', '¡Ve!'],
 ['Go.', 'Váyase.'],
 ['Go!', '¡Fuera!'],
 ['Go!', '¡Ya!'],
 ['So?', '¿Entonces?'],
 ['Go!', '¡Sal!'],
 ['Go.', 'Vaya.'],
 ['Me?', '¿Yo?'],
 ['Hi.', 'Hola.'],
 ['So?', '¿Y?']]

In [17]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [18]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go!', 'Go.', 'Go!', 'Go!', 'So?', 'Go!', 'Go.', 'Me?', 'Hi.', 'So?']
['¡Ve!', 'Váyase.', '¡Fuera!', '¡Ya!', '¿Entonces?', '¡Sal!', 'Vaya.', '¿Yo?', 'Hola.', '¿Y?']


In [19]:
def preprocess_sentence(sentence):
    """
        This function remove spanish accents and add to any sentence the start of sentence (<sos>) and end of sentence (<eos>) to all sentences in the data.
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [20]:
s1 = '¿Hola @ cómo estás? 123'

In [21]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [22]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [23]:
eng_sentences

['<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> so <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> me <eos>',
 '<sos> hi <eos>',
 '<sos> so <eos>',
 '<sos> hi <eos>',
 '<sos> go <eos>',
 '<sos> hi <eos>',
 '<sos> no <eos>',
 '<sos> ok <eos>',
 '<sos> ok <eos>',
 '<sos> ok <eos>',
 '<sos> so <eos>',
 '<sos> ow <eos>',
 '<sos> go <eos>',
 '<sos> no <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> go <eos>',
 '<sos> ah <eos>',
 '<sos> go <eos>',
 '<sos> see <eos>',
 '<sos> cut <eos>',
 '<sos> try <eos>',
 '<sos> run <eos>',
 '<sos> shh <eos>',
 '<sos> try <eos>',
 '<sos> hey <eos>',
 '<sos> shh <eos>',
 '<sos> run <eos>',
 '<sos> eat <eos>',
 '<sos> see <eos>',
 '<sos> eat <eos>',
 '<sos> hey <eos>',
 '<sos> try <eos>',
 '<sos> now <eos>',
 '<sos> fly <eos>',
 '<sos> die <eos>',
 '<sos> bye <eos>',
 '<sos> bye <eos>',
 '<sos> bye <eos>',
 '<sos> wow <eos>',
 '<sos> yes <eos>',
 '<sos> yes <eos>',
 '<sos> run <eos>',
 '<sos>

In [24]:
spa_sentences[:10]

['<sos> ve <eos>',
 '<sos> vayase <eos>',
 '<sos> fuera <eos>',
 '<sos> ya <eos>',
 '<sos> entonces <eos>',
 '<sos> sal <eos>',
 '<sos> vaya <eos>',
 '<sos> yo <eos>',
 '<sos> hola <eos>',
 '<sos> y <eos>']

In [25]:
def build_vocab(sentences):
    """
        this function create the vocab and create a couple of dictionaries that will be used to map id to word and word to index.
        In addition it add to the dictionaries two additional tokens (pad and unk) to represent paddings and unknow characters.
    """
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [26]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [27]:
print(eng_vocab_size, spa_vocab_size)

28415 48285


In [28]:
class EngSpaDataset(Dataset):
    """
        This class inherits from Dataset and it will override the methods that allows to work with the dictionaries as a dataset.
        It also will allow to convert from sentence to index so this words can be converted to embeddings.
    """
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx
        
    def __len__(self):
        return len(self.eng_sentences)
    
    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]
        
        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

In [30]:
def collate_fn(batch):
    """
        This function adds some paddings words to the batch data. The expected input is the list of indexes 
        batch = tuple(english_torch, spanish_torch)
        The function returns new torch with sentences with the same length
    """
    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch] # I don't think this is doing something.
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch] # I don't think this is doing something.
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch
    

In [ ]:
def train(model, dataloader, loss_function, optimiser, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0 
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Decoder preprocessing
            target_input = spa_batch[:, :-1] 
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input) # Passing sentence in english and in spanish as inputs of the model
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output) # For the loss function we are passing the whole set of words that are in spanish as list of words (embeddings) that should learn.
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()
            
        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')
            
            

In [184]:
BATCH_SIZE = 64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [185]:
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [186]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

In [ ]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

In [ ]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!",
    "This is amazing!"
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)
